In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

import joblib

In [2]:
df = pd.read_csv("Expanded_Destinations.csv")
df.head()

,DestinationID,Name,State,Type,Popularity,BestTimeToVisit
0,1,Taj Mahal,Uttar Pradesh,Historical,8.691906,Nov-Feb
1,2,Goa Beaches,Goa,Beach,8.605032,Nov-Mar
2,3,Jaipur City,Rajasthan,City,9.225372,Oct-Mar
3,4,Kerala Backwaters,Kerala,Nature,7.977386,Sep-Mar
4,5,Leh Ladakh,Jammu and Kashmir,Adventure,8.399822,Apr-Jun


In [3]:
df = df.drop_duplicates()

df = df.dropna(
    subset=[
        "Name",
        "State",
        "Type",
        "Popularity",
        "BestTimeToVisit"
    ]
)

df = df.reset_index(drop=True)

In [4]:
print(df.shape)
print(df.isnull().sum())

(1000, 6)
DestinationID      0
Name               0
State              0
Type               0
Popularity         0
BestTimeToVisit    0
dtype: int64


In [5]:
sss = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)
for train_idx, test_idx in sss.split(df, df["Type"]):

    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()


In [6]:
print("Training data:", train_df.shape)
print("Testing data :", test_df.shape)

Training data: (800, 6)
Testing data : (200, 6)


In [7]:
feature_columns = [
    "State",
    "Type",
    "Popularity",
    "BestTimeToVisit"
]

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]

In [8]:
categorical_columns = [
    "State",
    "Type",
    "BestTimeToVisit"
]
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)


In [9]:
X_train_cat = encoder.fit_transform(
    X_train[categorical_columns]
)
X_test_cat = encoder.transform(
    X_test[categorical_columns]
)

In [10]:
scaler = StandardScaler()

X_train_popularity = scaler.fit_transform(
    X_train[["Popularity"]]
)

X_test_popularity = scaler.transform(
    X_test[["Popularity"]]
)

In [11]:
X_train_final = np.hstack([
    X_train_cat,
    X_train_popularity
])

X_test_final = np.hstack([
    X_test_cat,
    X_test_popularity
])

In [12]:
print(X_train_final.shape)
print(X_test_final.shape)

(800, 16)
(200, 16)


In [13]:
similarity_matrix = cosine_similarity(
    X_test_final,
    X_train_final
)

In [14]:
def recommend_destinations(
    state,
    place_type,
    best_time,
    top_n=5
):
    
    # User input
    user_data = pd.DataFrame({
        "State": [state],
        "Type": [place_type],
        "BestTimeToVisit": [best_time]
    })
    
    # Encode user categorical features
    user_cat = encoder.transform(
        user_data[categorical_columns]
    )
    
    # User popularity neutral value
    user_popularity = np.array([[0]])
    
    # Combine
    user_vector = np.hstack([
        user_cat,
        user_popularity
    ])
    
    # All destinations encode
    all_cat = encoder.transform(
        df[categorical_columns]
    )
    
    all_popularity = scaler.transform(
        df[["Popularity"]]
    )
    
    all_features = np.hstack([
        all_cat,
        all_popularity
    ])
    
    # Similarity
    similarities = cosine_similarity(
        user_vector,
        all_features
    )[0]
    
    # Copy dataframe
    result = df.copy()
    
    result["Similarity"] = similarities
    
    # Final score
    result["FinalScore"] = (
        0.7 * result["Similarity"] +
        0.3 * (
            result["Popularity"] /
            result["Popularity"].max()
        )
    )
    
    # Highest score first
    result = result.sort_values(
        "FinalScore",
        ascending=False
    )
    
    return result[
        [
            "DestinationID",
            "Name",
            "State",
            "Type",
            "Popularity",
            "BestTimeToVisit",
            "FinalScore"
        ]
    ].head(top_n)

In [15]:
recommend_destinations(
    state="Rajasthan",
    place_type="Historical",
    best_time="Oct-Mar",
    top_n=5
)

,DestinationID,Name,State,Type,Popularity,BestTimeToVisit,FinalScore
972,973,Jaipur City,Rajasthan,City,8.565934,Oct-Mar,0.736137
527,528,Jaipur City,Rajasthan,City,8.564172,Oct-Mar,0.736137
702,703,Jaipur City,Rajasthan,City,8.563698,Oct-Mar,0.736136
417,418,Jaipur City,Rajasthan,City,8.571570,Oct-Mar,0.736128
402,403,Jaipur City,Rajasthan,City,8.578601,Oct-Mar,0.736096


In [16]:
import joblib

joblib.dump(encoder, "destination_encoder.pkl")
joblib.dump(scaler, "popularity_scaler.pkl")

['popularity_scaler.pkl']

In [18]:
import pandas as pd
import numpy as np

# Load existing dataset
df = pd.read_csv("Expanded_Destinations.csv")

# -----------------------------
# Hotel Name according to State
# -----------------------------

hotel_names = {
    "Rajasthan": [
        "Royal Rajasthan Palace",
        "Pink City Heritage Hotel",
        "Jaipur Grand Hotel"
    ],
    
    "Goa": [
        "Goa Beach Resort",
        "Palm Paradise Resort",
        "Ocean View Goa Hotel"
    ],
    
    "Kerala": [
        "Kerala Backwater Resort",
        "Coconut Grove Hotel",
        "Kerala Nature Retreat"
    ],
    
    "Jammu and Kashmir": [
        "Himalayan View Resort",
        "Kashmir Valley Hotel",
        "Snow Peak Resort"
    ],
    
    "Uttar Pradesh": [
        "Taj Heritage Hotel",
        "Agra Palace Hotel",
        "Mughal View Resort"
    ]
}

# Default hotels for states not present above
default_hotels = [
    "Grand Travel Hotel",
    "City View Hotel",
    "Comfort Stay Resort"
]


# -----------------------------
# Assign Hotel Name
# -----------------------------

np.random.seed(42)

def get_hotel_name(state):
    
    if state in hotel_names:
        return np.random.choice(
            hotel_names[state]
        )
    
    return np.random.choice(
        default_hotels
    )


df["HotelName"] = df["State"].apply(
    get_hotel_name
)


# -----------------------------
# Hotel Price according to State
# -----------------------------

price_ranges = {
    "Rajasthan": (1800, 4500),
    "Goa": (2500, 6000),
    "Kerala": (2200, 5500),
    "Jammu and Kashmir": (2500, 6500),
    "Uttar Pradesh": (1500, 4000)
}


def get_hotel_price(state):
    
    if state in price_ranges:
        
        low, high = price_ranges[state]
        
        return np.random.randint(
            low,
            high + 1
        )
    
    return np.random.randint(
        1500,
        4001
    )


df["HotelPricePerNight"] = df["State"].apply(
    get_hotel_price
)


# -----------------------------
# Save Updated Dataset
# -----------------------------

df.to_csv(
    "Expanded_Destinations.csv",
    index=False
)


print("Dataset updated successfully!")

print(
    df[
        [
            "Name",
            "State",
            "HotelName",
            "HotelPricePerNight"
        ]
    ].head(10)
)

Dataset updated successfully!
                Name              State                 HotelName  \
0          Taj Mahal      Uttar Pradesh        Mughal View Resort   
1        Goa Beaches                Goa          Goa Beach Resort   
2        Jaipur City          Rajasthan        Jaipur Grand Hotel   
3  Kerala Backwaters             Kerala     Kerala Nature Retreat   
4         Leh Ladakh  Jammu and Kashmir     Himalayan View Resort   
5          Taj Mahal      Uttar Pradesh        Taj Heritage Hotel   
6        Goa Beaches                Goa      Ocean View Goa Hotel   
7        Jaipur City          Rajasthan  Pink City Heritage Hotel   
8  Kerala Backwaters             Kerala     Kerala Nature Retreat   
9         Leh Ladakh  Jammu and Kashmir          Snow Peak Resort   

   HotelPricePerNight  
0                2258  
1                5446  
2                2042  
3                4790  
4                3563  
5                1728  
6                5095  
7                2